In [ ]:
# =============================================================
# OpenQSim Phase 0A — RESUME from checkpoint 590
# =============================================================
# HOW TO USE:
#   1. Start a new Kaggle session with P100 GPU (recommended)
#      or T4 x2 if P100 unavailable.
#   2. Upload your checkpoint + raw data:
#        - /kaggle/working/checkpoints/sweep_checkpoint.json
#        - /kaggle/working/checkpoints/completed_combinations.txt
#        - /kaggle/working/benchmark_outputs/raw/*.json  (all 590 files)
#   3. Add Kaggle Secrets: KAGGLE_USERNAME, KAGGLE_KEY,
#        NVIDIA_API_KEY, NVIDIA_API_KEY_1 (optional), GROQ_API_KEY (optional)
#   4. Create your dataset at kaggle.com/datasets/add
#        name it exactly: openqsim-benchmarks
#   5. Run this cell.
# =============================================================

import os, subprocess, sys, shutil, json
from pathlib import Path

# ── 1. Install dependencies ───────────────────────────────────────────────
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q',
    'qiskit>=2.0,<3.0', 'qiskit-aer>=0.17', 'pynvml', 'nvidia-ml-py',
    'pyyaml', 'pandas', 'kaggle', 'requests', 'python-dotenv', 'networkx',
])
print('[OK] Dependencies installed')

# ── 2. Clone latest code ─────────────────────────────────────────────────
REPO_URL = 'https://github.com/lekkalaharsha/opensim-ai'
REPO_DIR = '/kaggle/working/OpenQSim'
if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)
subprocess.check_call(['git', 'clone', REPO_URL, REPO_DIR])
print('[OK] Cloned repo (QAOA fix + Phase 2 features)')
sys.path.insert(0, REPO_DIR)

# ── 3. Load secrets ──────────────────────────────────────────────────────
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    def _get_secret(name, required=True):
        try:
            val = secrets.get_secret(name)
            if val:
                os.environ[name] = val
                print(f'[OK] {name} loaded')
                return val
        except Exception:
            pass
        if required: print(f'WARN {name} not found in Kaggle Secrets')
        return ''
    _get_secret('KAGGLE_USERNAME'); _get_secret('KAGGLE_KEY')
    _get_secret('NVIDIA_API_KEY'); _get_secret('NVIDIA_API_KEY_1', required=False)
    _get_secret('GROQ_API_KEY', required=False)
except Exception as e:
    print(f'WARN secrets: {e}')

if os.environ.get('NVIDIA_API_KEY_1'):
    os.environ['NVIDIA_API_KEY_COUNT'] = '2'
    print('[OK] Dual NVIDIA key mode')

KAGGLE_USERNAME = os.environ.get('KAGGLE_USERNAME', '')
KAGGLE_DATASET  = f'{KAGGLE_USERNAME}/openqsim-benchmarks'
OUTPUT_DIR      = '/kaggle/working/benchmark_outputs'
CKPT_DIR        = '/kaggle/working/checkpoints'
SWEEP_CONFIG    = f'{REPO_DIR}/benchmark/sweep_config_0a.yaml'
print(f'[OK] Target dataset: {KAGGLE_DATASET}')

# ── 4. Remove failed QAOA records so they re-run with the fixed generator ─
subprocess.check_call([
    sys.executable,
    f'{REPO_DIR}/scripts/fix_failed_qaoa.py',
    '--raw-dir',  f'{OUTPUT_DIR}/raw',
    '--ckpt-dir', CKPT_DIR,
])

# ── 5. Show checkpoint state after cleanup ────────────────────────────────
ckpt_path = f'{CKPT_DIR}/sweep_checkpoint.json'
if os.path.exists(ckpt_path):
    with open(ckpt_path) as f: ckpt = json.load(f)
    print(f'[OK] Checkpoint: {ckpt.get("completed_count",0)} records done, '
          f'resuming from index {ckpt.get("last_completed_index",-1)+1}')
else:
    print('WARN No checkpoint — starting from record 0')
raw_count = len(list(Path(f'{OUTPUT_DIR}/raw').glob('*.json'))) \
    if Path(f'{OUTPUT_DIR}/raw').exists() else 0
print(f'[OK] Raw records on disk: {raw_count}')

# ── 6. Warn if dataset doesn't exist on Kaggle ───────────────────────────
from kaggle.api_client import KaggleAPIClient
if KaggleAPIClient(KAGGLE_DATASET).dataset_exists():
    print(f'[OK] Dataset exists: {KAGGLE_DATASET}')
else:
    print(f'WARN Dataset not found: {KAGGLE_DATASET}\n'
          f'     Create it at kaggle.com/datasets/add — sweep will zip as fallback')

# ── 7. Run the sweep (resumes automatically from checkpoint) ─────────────
from kaggle.runner import KaggleRunner
runner = KaggleRunner(
    sweep_config_path=SWEEP_CONFIG,
    output_dir=OUTPUT_DIR,
    checkpoint_interval=10,
    artifact_interval=50,
    kaggle_dataset=KAGGLE_DATASET,
    use_advisor=True,
)
sweep_result = runner.run()
print(f'\nSweep done: {sweep_result.completed_count}/{sweep_result.total_combinations}')
print(f'  OOM: {sweep_result.oom_count}  Errors: {sweep_result.error_count}')
print(f'  Time: {sweep_result.duration_seconds/3600:.1f}h')

# ── 8. Assemble dataset ───────────────────────────────────────────────────
from kaggle.dataset_assembler import DatasetAssembler
manifest = DatasetAssembler(
    raw_dir=f'{OUTPUT_DIR}/raw',
    dataset_dir=f'{OUTPUT_DIR}/datasets/openqsim_v0.1-small',
).assemble()
print(f'\nAssembled: {manifest.records} records, {manifest.successful_runs} successful')
print(f'  Backends: {manifest.backends}  Qubits: {manifest.qubit_range}')

# ── 9. Push to Kaggle Dataset (zip fallback on 403) ──────────────────────
dataset_dir = Path(OUTPUT_DIR)
(dataset_dir / 'dataset-metadata.json').write_text(json.dumps({
    'title': 'OpenQSim Benchmark Outputs',
    'id': KAGGLE_DATASET,
    'licenses': [{'name': 'MIT'}]
}))
result = subprocess.run(
    ['kaggle', 'datasets', 'version', '-p', str(dataset_dir),
     '-m', f'Phase 0A complete: {manifest.records} records'],
    capture_output=True, text=True
)
if result.returncode == 0:
    print(f'[OK] Pushed to {KAGGLE_DATASET}')
else:
    zip_path = shutil.make_archive('/kaggle/working/benchmark_outputs_final', 'zip', OUTPUT_DIR)
    print(f'WARN Push failed: {result.stderr[:200]}')
    print(f'[OK] Zip saved: {zip_path} — download from Output tab')

print('\n=== Phase 0A complete ===')
